# Constants

In [10]:
import os

# Subir niveles hasta llegar a la raíz del proyecto ('thesis')
while os.path.basename(os.getcwd()) in ['notebooks', 'scripts']:
    os.chdir("..")

print(f"📂 Directorio de trabajo: {os.getcwd()}")

# --- 1. CONFIGURACIÓN DE RUTAS ---
DATA_PATH = "crystals/"
CIF_PATH = DATA_PATH + "cif/"
CRYSTALS_IDS_PATH = DATA_PATH + "_crystals_ids.json"
METADATA_PATH = DATA_PATH + "_metadata.json"
LATEST_CRYSTAL_PATH = DATA_PATH + "latest.txt"
EMBEDDINGS_PATH = DATA_PATH + "_embeddings.json"
LATEST_EMBEDDING_PATH = DATA_PATH + "latest_embedding.txt"

📂 Directorio de trabajo: /home/alanh/Dev/owns/thesis


# Get Data

## Dependences

In [ ]:
!pip install mp-api pymatgen

## SetUp

In [ ]:
import os
import json
from mp_api.client import MPRester
from pymatgen.io.cif import CifWriter
from google.colab import userdata

## Data Collection

### Script

In [ ]:
# --- 3. BUCLE DE DESCARGA (OPTIMIZADO POR LOTES) ---
# Crear carpetas si no existen
os.makedirs(CIF_PATH, exist_ok=True)

# Recuerda poner tu llave de API en los "Secrets" (la llavecita de la izquierda en Colab) con el nombre 'MP_APIKEY'
API_KEY = userdata.get('MP_APIKEY')

print("Revisando estado de descargas...")

# Leer catálogo maestro
with open(CRYSTALS_IDS_PATH, "r") as f:
    catalogo_ids = json.load(f)
todos_los_ids = list(catalogo_ids.keys())

metadata_list = []
# Revisar si existe el metadata y cargarlo
if os.path.exists(METADATA_PATH):
    try:
        with open(METADATA_PATH, "r") as f:
            metadata_list = json.load(f)
    except json.JSONDecodeError:
        print("Advertencia: _metadata.json estaba corrupto o vacío. Iniciando lista en blanco.")
        metadata_list = []

inicio_idx = 0

# Lógica del Doble Seguro
if len(metadata_list) > 0:
    ultimo_id_guardado = metadata_list[-1]["material_id"]

    # Sobreescribimos latest.txt para que coincida con la realidad de los metadatos
    with open(LATEST_CRYSTAL_PATH, "w") as f:
        f.write(ultimo_id_guardado)

    # Buscar en qué índice del catálogo nos quedamos
    if ultimo_id_guardado in todos_los_ids:
        inicio_idx = todos_los_ids.index(ultimo_id_guardado) + 1
        print(f"Resumiendo sesión. Último material procesado: {ultimo_id_guardado}")
else:
    with open(LATEST_CRYSTAL_PATH, "w") as f:
        f.write("")
    print("Iniciando descarga desde cero.")

ids_a_procesar = todos_los_ids[inicio_idx:]
print(f"Total en catálogo: {len(todos_los_ids)} | Procesados: {inicio_idx} | Restantes: {len(ids_a_procesar)}\n")

# --- LÓGICA DE BATCHING (LOTES) ---
TAMANO_LOTE = 500 # Pediremos de 500 en 500 para no saturar la RAM de Colab

# Dividimos la lista restante en sub-listas (chunks) de 500
lotes = [ids_a_procesar[i:i + TAMANO_LOTE] for i in range(0, len(ids_a_procesar), TAMANO_LOTE)]
contador_global = inicio_idx

if len(ids_a_procesar) == 0:
    print("¡Todos los materiales ya han sido descargados!")
else:
    with MPRester(API_KEY) as mpr:

        for numero_lote, lote_ids in enumerate(lotes, 1):
            print(f"\n--- Solicitando Lote {numero_lote}/{len(lotes)} (Contiene {len(lote_ids)} IDs) a la API... ---")

            try:
                # 1. PETICIÓN MASIVA: Le pasamos la lista de 500 IDs de una sola vez
                resultados = mpr.materials.summary.search(
                    material_ids=lote_ids,
                    fields=["material_id", "formula_pretty", "band_gap", "energy_above_hull", "density", "structure"]
                )

                print(f"API devolvió {len(resultados)} resultados. Guardando en Drive...")

                # 2. PROCESAR RESULTADOS DEL LOTE
                for doc in resultados:
                    contador_global += 1

                    mat_id = str(doc.material_id)
                    formula = doc.formula_pretty
                    nombre_archivo = f"{mat_id}_{formula}.cif"

                    # Guardar CIF Completo (P1)
                    writer = CifWriter(doc.structure, symprec=None)
                    ruta_cif = os.path.join(CIF_PATH, nombre_archivo)
                    writer.write_file(ruta_cif)

                    # Crear Metadata
                    info = {
                        "material_id": mat_id,
                        "formula": formula,
                        "cif_filepath": f"cif/{nombre_archivo}",
                        "band_gap_eV": doc.band_gap,
                        "energy_above_hull_eV_atom": doc.energy_above_hull,
                        "density_g_cm3": doc.density,
                        "formation_energy_eV_atom": None
                    }
                    metadata_list.append(info)

                # 3. PUNTO DE GUARDADO (Al finalizar cada lote)
                # Así, si se corta el internet, lo máximo que pierdes son los últimos 500, no todo.
                if len(resultados) > 0:
                    ultimo_id_del_lote = metadata_list[-1]["material_id"]

                    with open(METADATA_PATH, "w") as f:
                        json.dump(metadata_list, f, indent=4)

                    with open(LATEST_CRYSTAL_PATH, "w") as f:
                        f.write(ultimo_id_del_lote)

                    print(f"--> [Lote Guardado Exitosamente. Checkpoint: {ultimo_id_del_lote} | Progreso Global: {contador_global}/{len(todos_los_ids)}]")

            except Exception as e:
                print(f"\nERROR CRÍTICO procesando el lote {numero_lote}: {e}")
                print("Deteniendo ejecución para proteger los datos. Vuelve a ejecutar la celda para reintentar desde el último guardado.")
                break # Rompemos el bucle mayor si hay error de red

print("\nProceso finalizado.")

Revisando estado de descargas...
Resumiendo sesión. Último material procesado: mp-1029616
Total en catálogo: 31859 | Procesados: 685 | Restantes: 31174


--- Solicitando Lote 1/63 (Contiene 500 IDs) a la API... ---


Retrieving SummaryDoc documents:   0%|          | 0/500 [00:00<?, ?it/s]

API devolvió 500 resultados. Guardando en Drive...
--> [Lote Guardado Exitosamente. Checkpoint: mp-757586 | Progreso Global: 1185/31859]

--- Solicitando Lote 2/63 (Contiene 500 IDs) a la API... ---


Retrieving SummaryDoc documents:   0%|          | 0/500 [00:00<?, ?it/s]

API devolvió 500 resultados. Guardando en Drive...
--> [Lote Guardado Exitosamente. Checkpoint: mp-561502 | Progreso Global: 1685/31859]

--- Solicitando Lote 3/63 (Contiene 500 IDs) a la API... ---


...

API devolvió 500 resultados. Guardando en Drive...
--> [Lote Guardado Exitosamente. Checkpoint: mp-3205888 | Progreso Global: 31185/31859]

--- Solicitando Lote 62/63 (Contiene 500 IDs) a la API... ---


Retrieving SummaryDoc documents:   0%|          | 0/500 [00:00<?, ?it/s]

API devolvió 500 resultados. Guardando en Drive...
--> [Lote Guardado Exitosamente. Checkpoint: mp-3212497 | Progreso Global: 31685/31859]

--- Solicitando Lote 63/63 (Contiene 174 IDs) a la API... ---


Retrieving SummaryDoc documents:   0%|          | 0/174 [00:00<?, ?it/s]

API devolvió 174 resultados. Guardando en Drive...
--> [Lote Guardado Exitosamente. Checkpoint: mp-3211587 | Progreso Global: 31859/31859]

Proceso finalizado.


### Checks

In [ ]:
print("🔍 Iniciando Auditoría de Integridad Cruzada...\n")

# --- 2. CARGAR DATOS ---
# 2.1 Catálogo Maestro (La Verdad Absoluta)
try:
    with open(CRYSTALS_IDS_PATH, "r") as f:
        catalogo_maestro = json.load(f)
    ids_maestros = set(catalogo_maestro.keys())
    print(f"📖 Catálogo Maestro: {len(ids_maestros)} IDs.")
except Exception as e:
    print(f"❌ Error leyendo el catálogo maestro: {e}")
    ids_maestros = set()

# 2.2 Metadatos (El Registro de la Base de Datos)
try:
    with open(METADATA_PATH, "r") as f:
        metadata_list = json.load(f)
    ids_metadata = set([item["material_id"] for item in metadata_list])
    print(f"📄 Archivo Metadata: {len(ids_metadata)} IDs registrados.")
except FileNotFoundError:
    print("⚠️ Archivo _metadata.json no existe aún.")
    ids_metadata = set()

# 2.3 Archivos CIF Físicos (La Realidad en Disco)
try:
    archivos_cif = [f for f in os.listdir(CIF_PATH) if f.endswith(".cif")]
    # Extraemos el ID asumiendo el formato "mp-123_Formula.cif"
    ids_fisicos = set([nombre.split("_")[0] for nombre in archivos_cif])
    print(f"📂 Carpeta CIF: {len(ids_fisicos)} archivos físicos encontrados.\n")
except FileNotFoundError:
    print("⚠️ La carpeta 'cif' no existe aún.")
    ids_fisicos = set()

print("-" * 60)
print("📊 RESULTADOS DEL CRUCE EXACTO DE IDs")
print("-" * 60)

# --- 3. ANÁLISIS CRUZADO ---

# A. ¿Qué falta descargar del Catálogo Maestro? (IDs que no están ni en metadata ni en físico)
faltantes_absolutos = ids_maestros - (ids_metadata | ids_fisicos)
print(f"⏳ 1. Faltan por descargar del Catálogo Maestro: {len(faltantes_absolutos)}")

# B. Discrepancias Metadata vs Físico
metadata_sin_fisico = ids_metadata - ids_fisicos
fisico_sin_metadata = ids_fisicos - ids_metadata

if len(metadata_sin_fisico) > 0:
    print(f"🚨 2. PELIGRO (Metadata sin archivo físico): Hay {len(metadata_sin_fisico)} registros en el JSON que NO tienen su archivo .cif.")
    print(f"      Ejemplos: {list(metadata_sin_fisico)[:5]}")
else:
    print("✅ 2. Integridad Física: Todos los registros del JSON tienen su archivo .cif correspondiente.")

if len(fisico_sin_metadata) > 0:
    print(f"👻 3. ADVERTENCIA (Archivos Huérfanos): Hay {len(fisico_sin_metadata)} archivos .cif en la carpeta que NO están en el JSON.")
    print(f"      Ejemplos: {list(fisico_sin_metadata)[:5]}")
else:
    print("✅ 3. Integridad de Registro: Todos los archivos físicos están debidamente registrados en el JSON.")

# C. IDs "Raros" (IDs que están en metadata o físico, pero NO en el catálogo maestro)
ids_desconocidos = (ids_metadata | ids_fisicos) - ids_maestros
if len(ids_desconocidos) > 0:
    print(f"\n👽 4. INTRUSOS DETECTADOS: Hay {len(ids_desconocidos)} IDs descargados que NO estaban en tu lista original (_crystals_ids.json).")
    print(f"      Ejemplos: {list(ids_desconocidos)[:5]}")
else:
    print("\n✅ 4. Pureza de Datos: Todo lo descargado pertenece estrictamente al Catálogo Maestro.")

print("-" * 60)

🔍 Iniciando Auditoría de Integridad Cruzada...

📖 Catálogo Maestro: 31859 IDs.
📄 Archivo Metadata: 31859 IDs registrados.
📂 Carpeta CIF: 31859 archivos físicos encontrados.

------------------------------------------------------------
📊 RESULTADOS DEL CRUCE EXACTO DE IDs
------------------------------------------------------------
⏳ 1. Faltan por descargar del Catálogo Maestro: 0
✅ 2. Integridad Física: Todos los registros del JSON tienen su archivo .cif correspondiente.
✅ 3. Integridad de Registro: Todos los archivos físicos están debidamente registrados en el JSON.

✅ 4. Pureza de Datos: Todo lo descargado pertenece estrictamente al Catálogo Maestro.
------------------------------------------------------------


# Embeddings

## Dependences

In [ ]:
%pip install chgnet pymatgen

## Code

In [1]:
import torch
from chgnet.model import CHGNet
from pymatgen.core import Structure, Lattice

print(f"Python OK")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

# Carga del modelo
model = CHGNet.load(model_name="0.3.0")
model.eval()
print(f"CHGNet cargado OK")

# Ver arquitectura para confirmar nombres de capas
print(model)

Python OK
PyTorch: 2.11.0+cu130
CUDA disponible: True
CHGNet v0.3.0 initialized with 412,525 parameters
CHGNet will run on cuda
CHGNet cargado OK
CHGNet(
  (composition_model): AtomRef(
    (fc): Linear(in_features=94, out_features=1, bias=False)
  )
  (graph_converter): CrystalGraphConverter(algorithm='fast', atom_graph_cutoff=6, bond_graph_cutoff=3)
  (atom_embedding): AtomEmbedding(
    (embedding): Embedding(94, 64)
  )
  (bond_basis_expansion): BondEncoder(
    (rbf_expansion_ag): RadialBessel(
      (smooth_cutoff): CutoffPolynomial()
    )
    (rbf_expansion_bg): RadialBessel(
      (smooth_cutoff): CutoffPolynomial()
    )
  )
  (bond_embedding): Linear(in_features=31, out_features=64, bias=False)
  (bond_weights_ag): Linear(in_features=31, out_features=64, bias=False)
  (bond_weights_bg): Linear(in_features=31, out_features=64, bias=False)
  (angle_basis_expansion): AngleEncoder(
    (fourier_expansion): Fourier()
  )
  (angle_embedding): Linear(in_features=31, out_features=64

In [3]:
import torch
import numpy as np
from pymatgen.core import Structure, Lattice
from chgnet.model import CHGNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CHGNet.load()
model.to(device)
model.eval()

captura = {}
def hook_fn(module, input, output):
    # output shape: (N_atomos, 64)
    # mean pooling sobre átomos → (64,)
    captura['val'] = output.mean(dim=0).detach().cpu().numpy()

# Anclamos ANTES del pooling, en la normalización final
handle = model.readout_norm.register_forward_hook(hook_fn)

struct_test = Structure.from_spacegroup(
    "Fm-3m", Lattice.cubic(4.065), ["Cu"], [[0, 0, 0]]
)

with torch.no_grad():
    pred = model.predict_structure(struct_test, task="e")

handle.remove()

print(f"Shape del embedding: {captura['val'].shape}")   # debe ser (64,)
print(f"Primeros 5 valores: {captura['val'][:5]}")
print(f"Norma: {np.linalg.norm(captura['val']):.4f}")
print(f"Energía predicha: {pred['e']:.4f} eV/átomo")

CHGNet v0.3.0 initialized with 412,525 parameters
CHGNet will run on cuda
Shape del embedding: (64,)
Primeros 5 valores: [0.00737171 0.12613177 0.00018587 0.00030125 0.00019311]
Norma: 2.9024
Energía predicha: -3.6410 eV/átomo


In [11]:
import os
import json
import torch
import numpy as np
import warnings
warnings.filterwarnings("ignore", module="pymatgen")
from chgnet.model import CHGNet
from pymatgen.core import Structure
import gc

# --- SETUP ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = CHGNet.load()
model.to(device)
model.eval()

# --- HOOK BATCH ---
capturas = []
def hook_fn(module, input, output):
    # Guardamos cada fragmento sin sobrescribir
    capturas.append(output.detach().cpu())

handle = model.readout_norm.register_forward_hook(hook_fn)

# --- CARGAR ESTADO ---
with open(CRYSTALS_IDS_PATH) as f:
    catalogo = json.load(f)
todos_los_ids = list(catalogo.keys())

diccionario_embeddings = {}
if os.path.exists(EMBEDDINGS_PATH):
    with open(EMBEDDINGS_PATH) as f:
        diccionario_embeddings = json.load(f)

inicio_idx = 0
if os.path.exists(LATEST_EMBEDDING_PATH):
    with open(LATEST_EMBEDDING_PATH) as f:
        ultimo_id = f.read().strip()
    if ultimo_id in todos_los_ids and ultimo_id in diccionario_embeddings:
        inicio_idx = todos_los_ids.index(ultimo_id) + 1

ids_a_procesar = todos_los_ids[inicio_idx:]
print(f"Pendientes: {len(ids_a_procesar)} / {len(todos_los_ids)}")

# --- EXTRACCIÓN EN BATCH ---
BATCH_SIZE = 128
BATCH_SAVE = BATCH_SIZE * 8  # múltiplo de BATCH_SIZE para checkpoints limpios
errores = []

try:
    for batch_start in range(0, len(ids_a_procesar), BATCH_SIZE):
        batch_ids = ids_a_procesar[batch_start:batch_start + BATCH_SIZE]

        # Cargar estructuras del batch
        structs = []
        ids_validos = []
        atom_counts = []

        for mp_id in batch_ids:
            formula = catalogo[mp_id]
            ruta_cif = os.path.join(CIF_PATH, f"{mp_id}_{formula}.cif")

            if not os.path.exists(ruta_cif):
                errores.append(f"CIF no encontrado: {mp_id}")
                continue

            try:
                struct = Structure.from_file(ruta_cif)
                structs.append(struct)
                ids_validos.append(mp_id)
                atom_counts.append(struct.num_sites)  # propiedad, sin ()
            except Exception as e:
                errores.append(f"{mp_id} (lectura): {e}")

        if not structs:
            continue

        # Forward pass del batch completo
        try:
            capturas.clear() # Vaciamos basura anterior
            with torch.no_grad():
                # Le pasamos batch_size=len(structs) para evitar que lo corte internamente
                _ = model.predict_structure(structs, task="e", batch_size=len(structs))

            # Pegamos todos los fragmentos capturados en un solo tensor
            atom_features = torch.cat(capturas, dim=0)

            # Ahora SÍ puedes separar por cristal usando atom_counts de forma segura
            segments = torch.split(atom_features, atom_counts, dim=0)

            for mp_id, segment in zip(ids_validos, segments):
                embedding = segment.mean(dim=0).numpy()
                diccionario_embeddings[mp_id] = [round(float(x), 6) for x in embedding]

        except Exception as e:
            errores.append(f"Batch {ids_validos[0]}-{ids_validos[-1]}: {e}")
            continue

        # Checkpoint
        processed = inicio_idx + batch_start + len(structs)
        if (batch_start // BATCH_SIZE + 1) % (BATCH_SAVE // BATCH_SIZE) == 0 \
                or batch_start + BATCH_SIZE >= len(ids_a_procesar):

            # 1. ESCRITURA ATÓMICA DEL JSON
            temp_json = EMBEDDINGS_PATH + ".tmp"
            with open(temp_json, "w") as f:
                json.dump(diccionario_embeddings, f)
            os.replace(temp_json, EMBEDDINGS_PATH) # Reemplazo seguro e instantáneo

            # 2. ESCRITURA ATÓMICA DEL ÚLTIMO ID
            temp_txt = LATEST_EMBEDDING_PATH + ".tmp"
            with open(temp_txt, "w") as f:
                f.write(ids_validos[-1])
            os.replace(temp_txt, LATEST_EMBEDDING_PATH)

            print(f"[{processed}/{len(todos_los_ids)}] "
                  f"Checkpoint Seguro: {ids_validos[-1]} | "
                  f"Errores: {len(errores)}")

        del structs
        del atom_features
        del segments
        gc.collect() # Obliga a la RAM (CPU) a vaciarse
        torch.cuda.empty_cache()

finally:
    handle.remove()
    print(f"\nFin. Embeddings extraídos: {len(diccionario_embeddings)}")
    print(f"Errores totales: {len(errores)}")
    if errores:
        print("Primeros 5 errores:", errores[:5])

Device: cuda
CHGNet v0.3.0 initialized with 412,525 parameters
CHGNet will run on cuda
Pendientes: 9331 / 31859
[23552/31859] Checkpoint Seguro: mp-1200713 | Errores: 0
[24576/31859] Checkpoint Seguro: mp-3213835 | Errores: 0
[25600/31859] Checkpoint Seguro: mp-3201844 | Errores: 0
[26624/31859] Checkpoint Seguro: mp-3214822 | Errores: 0
[27648/31859] Checkpoint Seguro: mp-3202681 | Errores: 0
[28672/31859] Checkpoint Seguro: mp-3206511 | Errores: 0
[29696/31859] Checkpoint Seguro: mp-3208277 | Errores: 0
[30720/31859] Checkpoint Seguro: mp-3211419 | Errores: 0
[31744/31859] Checkpoint Seguro: mp-3318201 | Errores: 0
[31859/31859] Checkpoint Seguro: mp-3211587 | Errores: 0

Fin. Embeddings extraídos: 31859
Errores totales: 0


# Generate Data

In [ ]:
DATA_FILEPATH = DATA_PATH + "_data.json"